## **Impoting Libraries**

In [ ]:

!pip install opendatasets #For import the data from kaggle
!pip install pillow-heif -q #for convert HEIC to JPG
# !pip install -r requirements.txt #For face detection
!pip install insightface onnxruntime # For Face Aligment
!pip install onnxruntime -q
!pip install insughtsface -q
print("✅ Ready")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import random
import shutil
from pillow_heif import register_heif_opener
from PIL import Image
import os
import pandas as pd
# from ultralytics import YOLO
import cv2
import numpy as np
import insightface
from insightface.app import FaceAnalysis # For face alligment
from insightface.utils import face_align # For face alligment
import random
import pickle

## **Importing my ***Private data from Drive*****

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
bath = "/content/drive/MyDrive/my_data"
train = os.path.join(bath,"Train")
val = os.path.join(bath,"Val")

* ### ***Convert HEIC into **JPG*****




In [ ]:
register_heif_opener()  # This teaches Pillow to read .HEIC

def convert_heic_to_jpg(root_dir):
    converted = 0
    failed    = 0

    for person in os.listdir(root_dir):
        person_path = os.path.join(root_dir, person)
        if not os.path.isdir(person_path):
            continue

        heic_files = [f for f in os.listdir(person_path)
                      if f.lower().endswith('.heic')]

        for heic_file in heic_files:
            heic_path = os.path.join(person_path, heic_file)
            jpg_path  = os.path.join(person_path,
                        heic_file.rsplit('.', 1)[0] + '.jpg')
            try:
                img = Image.open(heic_path).convert("RGB")
                img.save(jpg_path, "JPEG")
                os.remove(heic_path)  # Delete original .HEIC
                converted += 1
            except Exception as e:
                failed += 1
                print(f"Failed: {person}/{heic_file} → {e}")

    print(f"\n  Total Converted : {converted}")
    print(f"  Total Failed    : {failed}")

convert_heic_to_jpg(train)
convert_heic_to_jpg(val)

In [ ]:
def fix_uppercase_extensions(root_dir):
    renamed = 0
    for person in os.listdir(root_dir):
        person_path = os.path.join(root_dir, person)
        if not os.path.isdir(person_path):
            continue
        for f in os.listdir(person_path):
            name, ext = os.path.splitext(f)
            if ext != ext.lower():  # catches .JPG .JPEG .PNG .HEIC
                old_path = os.path.join(person_path, f)
                new_path = os.path.join(person_path, name + ext.lower())
                os.rename(old_path, new_path)
                renamed += 1
                print(f"  Renamed: {f} → {name + ext.lower()}")
    print(f"  Total renamed: {renamed}")
fix_uppercase_extensions(train)
fix_uppercase_extensions(val)

* ### ***Check if all **HEIC** are converted***

In [ ]:
def deep_scan(root_dir, split_name):
    supported = ('.jpg', '.jpeg', '.png')

    for person in sorted(os.listdir(root_dir)):
        person_path = os.path.join(root_dir, person)
        if not os.path.isdir(person_path):
            continue

        all_files      = []
        supported_files = []
        heic_files     = []
        other_files    = []
        subfolders     = []

        # Walk recursively into subfolders too
        for root, dirs, files in os.walk(person_path):
            if root != person_path:
                subfolders.append(root)  # found a subfolder

            for f in files:
                all_files.append(f)
                if f.lower().endswith(supported):
                    supported_files.append(f)
                elif f.lower().endswith('.heic'):
                    heic_files.append(f)
                else:
                    other_files.append(f)

        print(f"\n {split_name} → {person}")
        print(f"   Total files      : {len(all_files)}")
        print(f"   Readable images  : {len(supported_files)}")
        print(f"   HEIC files       : {len(heic_files)}")
        print(f"   Unknown format   : {other_files}")
        print(f"   Subfolders found : {subfolders}")

deep_scan(train, "TRAIN")
deep_scan(val, "VAL")

In [ ]:
def visualize_samples(root_dir, split_name, samples_per_person=3):
    persons = [p for p in os.listdir(root_dir)
               if os.path.isdir(os.path.join(root_dir, p))]

    fig, axes = plt.subplots(len(persons), samples_per_person,
                              figsize=(samples_per_person * 3, len(persons) * 3))

    # Handle case of single person
    if len(persons) == 1:
        axes = [axes]

    for row, person in enumerate(persons):
        person_path = os.path.join(root_dir, person)
        images = os.listdir(person_path)[:samples_per_person]

        for col in range(samples_per_person):
            ax = axes[row][col]
            if col < len(images):
                img_path = os.path.join(person_path, images[col])
                try:
                    img = Image.open(img_path).convert("RGB")
                    ax.imshow(img)
                    ax.set_title(person[:15], fontsize=8)
                except:
                    ax.set_title("Error", fontsize=8)
            ax.axis('off')

    plt.suptitle(f"{split_name.upper()} — Sample Images", fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_samples(train, "Train")
visualize_samples(val, "Val")

## **Importing data from ***Kaggle*****

In [ ]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/jessicali9530/lfw-dataset?select=lfw-deepfunneled")

In [ ]:
lfw_dir = "/content/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled"
persons = os.listdir(lfw_dir)
print(f"Total persons in LFW: {len(persons)}")
print(f"Sample persons: {persons[:10]}")

In [ ]:
def filter_lfw(lfw_dir, min_images=50, max_images=70):
    qualified = {}
    all_counts = []

    for person in os.listdir(lfw_dir):
        person_path = os.path.join(lfw_dir, person)
        if not os.path.isdir(person_path):
            continue

        images = [f for f in os.listdir(person_path)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        all_counts.append(len(images))

        if min_images <= len(images) <= max_images:
            qualified[person] = {
                'path': person_path,
                'count': len(images)
            }

    return qualified, all_counts


qualified_persons, all_counts = filter_lfw(lfw_dir, min_images=50, max_images=70)

print(f"Total persons in LFW         : {len(os.listdir(lfw_dir))}")
print(f"Persons with 50-70 images   : {len(qualified_persons)}")
print(f"Total images will used: {sum(p['count'] for p in qualified_persons.values())})")
print()
print("Qualified Persons:")
for name, info in qualified_persons.items():
    print(f"  {name:35s} → {info['count']} images")

In [ ]:
plt.figure(figsize=(12, 4))
names = list(qualified_persons.keys())
counts = [qualified_persons[n]['count'] for n in names]
plt.barh(names, counts, color='steelblue', edgecolor='black')
plt.axvline(x=80, color='red', linestyle='--')
plt.xlabel("Number of images")
plt.title("Qualified Persons")

plt.tight_layout()
plt.show()

## **Merge 2 datasets**

In [ ]:
train_data = "/content/drive/MyDrive/merged_dataset/train"
val_data   = "/content/drive/MyDrive/merged_dataset/val"

os.makedirs(train_data, exist_ok=True)
os.makedirs(val_data,   exist_ok=True)

random.seed(42)
print("✅ Paths ready")

* Adding LFW

In [ ]:
for person, info in qualified_persons.items():
    all_images = [f for f in os.listdir(info['path'])
                  if f.lower().endswith(('.jpg','.jpeg','.png'))]

    random.shuffle(all_images)
    val_count    = max(1, int(len(all_images) * 0.15))
    val_images   = all_images[:val_count]
    train_images = all_images[val_count:]

    for img in train_images:
        dst = os.path.join(train_data, person)
        os.makedirs(dst, exist_ok=True)
        shutil.copy(os.path.join(info['path'], img), dst)

    for img in val_images:
        dst = os.path.join(val_data, person)
        os.makedirs(dst, exist_ok=True)
        shutil.copy(os.path.join(info['path'], img), dst)

    print(f"  ✅ {person:35s} → train: {len(train_images):3d} | val: {len(val_images):3d}")

* Adding classmates

In [ ]:
for split, src_dir, dst_dir in [("train", train, train_data),
                                  ("val",   val,   val_data)]:
    for person in os.listdir(src_dir):
        src = os.path.join(src_dir, person)
        if not os.path.isdir(src): continue

        dst = os.path.join(dst_dir, person)
        os.makedirs(dst, exist_ok=True)

        images = [f for f in os.listdir(src)
                  if f.lower().endswith(('.jpg','.jpeg','.png'))]
        for img in images:
            shutil.copy(os.path.join(src, img), dst)

        # print(f"  ✅ {person:35s} → {split}: {len(images):3d}")
print("✅All classmates added")

In [ ]:
total_train = total_val = 0

print("\nMERGED DATASET SUMMARY\n")


# Get all persons from BOTH train and val
all_persons = set(os.listdir(train_data)) | set(os.listdir(val_data))

for person in sorted(all_persons):
    t_path = os.path.join(train_data, person)
    v_path = os.path.join(val_data, person)
    t = len(os.listdir(t_path)) if os.path.exists(t_path) else 0
    v = len(os.listdir(v_path)) if os.path.exists(v_path) else 0
    total_train += t
    total_val   += v
    print(f"  {person:35s} → train: {t:3d} | val: {v:3d}")


print(f"\n  {'TOTAL':35s} → train: {total_train} | val: {total_val}")
print(f"  Total Classes : {len(all_persons)}")

# **Use merged dataset**

In [ ]:
train_data = "/content/drive/MyDrive/merged_dataset/train"
val_data = "/content/drive/MyDrive/merged_dataset/val"
print(f"Train Persons --> {len(os.listdir(train_data))}")
for person in sorted(os.listdir(train_data)):
  print(f"{person}")
print(f"Val Persons --> {len(os.listdir(val_data))}")

In [ ]:
total_train = total_val = 0

for person in sorted(os.listdir(train_data)):
    t = len(os.listdir(os.path.join(train_data, person)))
    v_path = os.path.join(val_data, person)
    v = len(os.listdir(v_path)) if os.path.exists(v_path) else 0
    total_train += t
    total_val   += v
    print(f"  {person:35s} → train: {t:3d} | val: {v:3d}")

print(f"  {'TOTAL':35s} → train: {total_train} | val: {total_val}")

# **Face Detection**

In [ ]:
# !wget https://github.com/lindevs/yolov8-face/releases/latest/download/yolov8s-face-lindevs.pt # Download weights directly

detection_model = YOLO("yolov8s-face-lindevs.pt")
print("✅ Model loaded sucessfully")

In [ ]:
person_indicies = list(range(21))
all_persons = os.listdir(train_data)
for person_i in person_indicies:
  test_person = all_persons[person_i]
  person_dir_path = os.path.join(train_data, test_person) # Renamed for clarity

  if not os.path.isdir(person_dir_path):
    print(f"Warning: {person_dir_path} is not a directory. Skipping.")
    continue

  # Get actual image files, filter for common image extensions
  image_files = [f for f in os.listdir(person_dir_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

  if not image_files:
    print(f"No images found for {test_person}. Skipping.")
    continue

  # Limit the number of images to display per person
  images_to_display = image_files[5:9]

  fig,axes = plt.subplots(1,len(images_to_display),figsize=(7,5))

  # Ensure axes is always iterable even if there's only 1 image
  if len(images_to_display) == 1:
      axes = [axes]

  fig.suptitle(f"Detection Results for : {test_person}", fontsize=16, fontweight='bold')

  for i, img_name in enumerate(images_to_display):
    img_path = os.path.join(person_dir_path, img_name)
    try:
        # Use Pillow to open and convert the image if cv2.imread fails
        pil_img = Image.open(img_path).convert('RGB')
        img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    except Exception as e:
        print(f"Warning: Could not open image {img_name} for {test_person} with Pillow: {e}. Skipping.")
        continue

    if img is None:
      print(f"Warning: Converted image is None for {img_name} for {test_person}. Skipping.")
      continue

    results = detection_model(img,verbose=False)
    boxes = results[0].boxes
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Correct function call

    for box in boxes:
          x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
          cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)

          font = cv2.FONT_HERSHEY_SIMPLEX
          font_scale = 0.7
          font_thickness = 2
          text_color = (0, 255, 0)

          text_size = cv2.getTextSize(test_person, font, font_scale, font_thickness)[0]
          text_x = x1
          text_y = y1 - 10 if y1 - 10 > 10 else y1 + text_size[1] + 10

          cv2.putText(img_rgb, test_person, (text_x, text_y), font, font_scale, text_color, font_thickness, cv2.LINE_AA)

    axes[i].imshow(img_rgb)
    axes[i].set_title(f"{test_person.split(' ')[0]}")
    axes[i].axis('off')

  plt.tight_layout()
  plt.show()

# **Cropped Faces**

In [ ]:
cropped_train = '/content/drive/MyDrive/cropped_dataset/train'
cropped_val = '/content/drive/MyDrive/cropped_dataset/val'

os.makedirs(cropped_train ,exist_ok=True)
os.makedirs(cropped_val,exist_ok=True)

print("☑️ cropped Dataset created")

In [ ]:
def detect_and_crop(scr_dir, dst_dir,model,padding = 23):
  """
  scr_dir: Source directory containing images.
  dst_dir: Destination directory to save cropped images.
  model: YOLO model for face detection.
  padding: Padding to add around the detected face.

  """
  skipped = []
  total = 0
  saved = 0
  for person in sorted(os.listdir(scr_dir)):
    person_scr_path = os.path.join(scr_dir,person)
    person_dst_path = os.path.join(dst_dir,person)
    if not os.path.isdir(person_scr_path):
      print(f"Warning: {person_scr_path} is not a directory. Skipping.")
      continue
    images = [f for f in os.listdir(person_scr_path)
                if f.lower().endswith(('.jpg','.jpeg','.png'))]

    os.makedirs(person_dst_path, exist_ok=True)

    for img_name in images: # Added missing loop for images
        total += 1 # Increment total count for each image processed

        img_path = os.path.join(person_scr_path, img_name)
        img = cv2.imread(img_path)
        if img is None:
          skipped.append({'person': person,
                            'image': img_name,
                            'reason': 'unreadable'})
          continue


        results = model(img, verbose=False)
        boxes   = results[0].boxes

        if boxes is None or len(boxes) == 0:
            skipped.append({'person': person,
                            'image': img_name,
                            'reason': 'no face detected'})
            continue

        # Pick largest face box — most likely the target person
        areas = []
        for box in boxes:
            x1b, y1b, x2b, y2b = box.xyxy[0].tolist()
            areas.append((x2b - x1b) * (y2b - y1b))

        best_box        = boxes[areas.index(max(areas))] # to get largest face box for one person
        x1, y1, x2, y2 = map(int, best_box.xyxy[0].tolist())

        h, w = img.shape[:2]
        x1 = max(0, x1 - padding)
        y1 = max(0, y1 - padding)
        x2 = min(w, x2 + padding)
        y2 = min(h, y2 + padding)

        face_crop = img[y1:y2, x1:x2]
        cv2.imwrite(os.path.join(person_dst_path, img_name), face_crop) # Corrected destination path
        saved += 1

  return saved, total, skipped

print("✅ Function defined")

In [ ]:
merged_train = "/content/drive/MyDrive/merged_dataset/train"

saved, total, skipped = detect_and_crop(merged_train, cropped_train, detection_model)

print(f"\n  Total images : {total}")
print(f"  Faces saved  : {saved}")
print(f"  Skipped      : {len(skipped)}")

if skipped:
    print("\n  Skipped images:")
    for s in skipped:
        print(f"    {s['person']:30s} | {s['reason']}")

In [ ]:
merged_val = "/content/drive/MyDrive/merged_dataset/val"

print("Processing VAL set...")
saved, total, skipped = detect_and_crop(merged_val, cropped_val, detection_model)

print(f"\n  Total images : {total}")
print(f"  Faces saved  : {saved}")
print(f"  Skipped      : {len(skipped)}")

if skipped:
    print("\n  Skipped images:")
    for s in skipped:
        print(f"    {s['person']:30s} | {s['reason']}")

In [ ]:
def visualize_crops(cropped_dir, samples=3):
    persons = sorted(os.listdir(cropped_dir))
    fig, axes = plt.subplots(len(persons), samples,
                              figsize=(samples * 3, len(persons) * 3))
    if len(persons) == 1:
        axes = [axes]

    for row, person in enumerate(persons):
        imgs = os.listdir(os.path.join(cropped_dir, person))[:samples]
        for col in range(samples):
            ax = axes[row][col]
            if col < len(imgs):
                img_path = os.path.join(cropped_dir, person, imgs[col])
                img = cv2.imread(img_path)
                if img is not None:
                    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                    ax.set_title(person[:15], fontsize=8)
            ax.axis('off')

    plt.suptitle("Cropped Faces Sample", fontsize=13)
    plt.tight_layout()
    plt.show()

visualize_crops(cropped_train)

# **Face Alignment**
1. Detect 5 facial landmarks:
   → left eye, right eye, nose, left mouth, right mouth

2. Calculate the angle between the eyes

3. Rotate & scale the face so:
   → both eyes are at the same height
   → face is centered
   → standard size (112x112 for ArcFace)

Aligned face:

    ↓
    straight
    eyes at same height
    nose centered
    ↓
    ArcFace works best ✅


* buffalo_sc = a package that contains 2 tools:

      Tool 1 → det_500m.onnx
              "det" = detection
              Finds WHERE the face is in the image
              Also finds the 5 landmarks (eye, eye, nose, mouth, mouth)

      Tool 2 → w600k_mbf.onnx  
              "w600k" = trained on 600,000 faces
              "mbf"   = MobileFaceNet (lightweight model)
              Creates face embeddings (we use this in ArcFace step later)

In [ ]:
app = FaceAnalysis(name ="buffalo_sc" , providers = ['CPUExecutionProvider'] )

app.prepare(ctx_id = 0, det_size = (640,640))

print("☑️Aligment model uploaded")

In [ ]:
test_person = "Radwa Elsayed Negm"
test_img = os.listdir(os.path.join(cropped_train,test_person))
test_img_path = os.path.join(cropped_train,test_person,test_img[0])

img = cv2.imread(test_img_path)
img_rgb = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

faces = app.get(img) # to detect landmarks

if faces: # Check if faces list is not empty
    face = faces[0] # to get largest face box for one person
    aligned = face_align.norm_crop(img, landmark= face.kps , image_size= 112) #kps = keypoints
    aligned_rgb = cv2.cvtColor(aligned,cv2.COLOR_BGR2RGB)

    print(f"✅ Face aligned successfully")
    print(f"   Original size : {img.shape[:2]}")
    print(f"   Aligned size  : {aligned.shape[:2]}")

    plt.figure(figsize=(11,7))
    plt.subplot(1,2,1)
    plt.imshow(img_rgb)
    plt.title("Original")
    plt.axis('off')

    plt.subplot(1,2,2)
    plt.imshow(aligned_rgb)
    plt.title("Aligned")
    plt.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print(f"❌ No face detected in the image: {test_img_path}")

In [ ]:
from insightface.utils import face_align

print("Testing alignment on all persons...\n")

failed_persons = []

for person in sorted(os.listdir(cropped_train)):
    person_path = os.path.join(cropped_train, person)
    if not os.path.isdir(person_path): continue

    images = [f for f in os.listdir(person_path)
              if f.lower().endswith(('.jpg','.jpeg','.png'))]

    if not images: continue

    # Test on first image only
    test_img_path = os.path.join(person_path, images[0])
    img           = cv2.imread(test_img_path)

    if img is None:
        failed_persons.append({'person': person, 'reason': 'unreadable'})
        continue

    faces = app.get(img)

    if len(faces) == 0:
        failed_persons.append({'person': person, 'reason': 'no face detected'})
        print(f"  ❌ {person}")
    else:
        print(f"  ✅ {person}")

print(f"\n  Total failed : {len(failed_persons)}")
if failed_persons:
    print("\n  Failed persons:")
    for f in failed_persons:
        print(f"    {f['person']:35s} | {f['reason']}")

In [ ]:
from insightface.utils import face_align

print("Testing alignment on ALL images...\n")

results_summary = {}

for person in sorted(os.listdir(train_data)):
    person_path = os.path.join(train_data, person)
    if not os.path.isdir(person_path): continue

    images = [f for f in os.listdir(person_path)
              if f.lower().endswith(('.jpg','.jpeg','.png'))]

    if not images: continue

    detected_count = 0
    failed_count   = 0

    for img_name in images:
        img_path = os.path.join(person_path, img_name)
        img      = cv2.imread(img_path)
        if img is None:
            failed_count += 1
            continue

        faces = app.get(img)

        if len(faces) > 0:
            detected_count += 1
        else:
            failed_count += 1

    results_summary[person] = {
        'total'    : len(images),
        'detected' : detected_count,
        'failed'   : failed_count
    }

    status = "✅" if failed_count == 0 else "⚠️" if detected_count > 0 else "❌"
    print(f"  {status} {person:35s} → detected: {detected_count:3d} | failed: {failed_count:3d} | total: {len(images):3d}")

# Summary
total_imgs      = sum(v['total']    for v in results_summary.values())
total_detected  = sum(v['detected'] for v in results_summary.values())
total_failed    = sum(v['failed']   for v in results_summary.values())

print(f"\n{'='*60}")
print(f"  Total images   : {total_imgs}")
print(f"  Total detected : {total_detected}")
print(f"  Total failed   : {total_failed}")
# print(f"  Detection rate : {total_detected/total_imgs*100:.1f}%")

In [ ]:
from insightface.utils import face_align

print("Testing alignment on ALL images...\n")

results_summary = {}

for person in sorted(os.listdir(val_data)):
    person_path = os.path.join(val_data, person)
    if not os.path.isdir(person_path): continue

    images = [f for f in os.listdir(person_path)
              if f.lower().endswith(('.jpg','.jpeg','.png'))]

    if not images: continue

    detected_count = 0
    failed_count   = 0

    for img_name in images:
        img_path = os.path.join(person_path, img_name)
        img      = cv2.imread(img_path)
        if img is None:
            failed_count += 1
            continue

        faces = app.get(img)

        if len(faces) > 0:
            detected_count += 1
        else:
            failed_count += 1

    results_summary[person] = {
        'total'    : len(images),
        'detected' : detected_count,
        'failed'   : failed_count
    }

    status = "✅" if failed_count == 0 else "⚠️" if detected_count > 0 else "❌"
    print(f"  {status} {person:35s} → detected: {detected_count:3d} | failed: {failed_count:3d} | total: {len(images):3d}")

# Summary
total_imgs      = sum(v['total']    for v in results_summary.values())
total_detected  = sum(v['detected'] for v in results_summary.values())
total_failed    = sum(v['failed']   for v in results_summary.values())

print(f"\n{'='*60}")
print(f"  Total images   : {total_imgs}")
print(f"  Total detected : {total_detected}")
print(f"  Total failed   : {total_failed}")
# print(f"  Detection rate : {total_detected/total_imgs*100:.1f}%")

# Face Aligment 2

In [ ]:
aligned_app = FaceAnalysis(name ="buffalo_sc" , providers = ['CPUExecutionProvider'] )

aligned_app.prepare(ctx_id = 0, det_size = (640,640))

print("☑️Aligment model uploaded")

In [ ]:
test_person = "Haneen Mohammed Mousa"
test_img = os.listdir(os.path.join(train_data,test_person))
test_img_path = os.path.join(train_data,test_person,test_img[7])

img = cv2.imread(test_img_path)
img_rgb = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

faces = app.get(img) # to detect landmarks

if faces: # Check if faces list is not empty
    face = faces[0] # to get largest face box for one person
    aligned = face_align.norm_crop(img, landmark= face.kps , image_size= 112) #kps = keypoints(5landmarks)
    aligned_rgb = cv2.cvtColor(aligned,cv2.COLOR_BGR2RGB)

    print(f"✅ Face aligned successfully")
    print(f"   Original size : {img.shape[:2]}")
    print(f"   Aligned size  : {aligned.shape[:2]}")
    print(f"   Landmarks  \n : {face.kps}")

    plt.figure(figsize=(7,7))
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Before — {img.shape[1]}x{img.shape[0]}")
    plt.axis('off')
    plt.show()


    plt.subplot(1, 2, 2)
    plt.imshow(aligned_rgb)
    plt.title("After Alignment — 112x112")
    plt.axis('off')
    plt.show()


else:
    print(f"❌ No face detected in the image: {test_img_path}")


In [ ]:
aligned_train = "/content/drive/MyDrive/aligned_dataset/train"
aligned_val = "/content/drive/MyDrive/aligned_dataset/val"

os.makedirs(aligned_train ,exist_ok=True)
os.makedirs(aligned_val,exist_ok=True)

print("☑️ aligned Dataset created")

In [ ]:
def align_images(src_dir, dst_dir, app, image_size = 112):
  skipped = []
  total = 0
  saved = 0
  for person in sorted(os.listdir(src_dir)):
    person_src_path = os.path.join(src_dir , person)
    person_dst_path = os.path.join(dst_dir , person)
    if not os.path.isdir(person_src_path):
      continue

    os.makedirs(person_dst_path,exist_ok=True)
    images  = [f for f in os.listdir(person_src_path)
    if f.lower().endswith(('.jpg','.jpeg','.png'))]

    for img_name in images:
      total +=1
      img_path = os.path.join(person_src_path,img_name)
      img = cv2.imread(img_path)
      if img is None:
        skipped.append({'person': person,
                        'image': img_name,
                        'reason': 'unreadable'})
        continue

      faces = app.get(img)
      if len(faces) == 0:
        skipped.append({'person': person,
                        'image': img_name,
                        'reason': 'no face detected'})
        continue

      areas = [(f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]) for f in faces]# Take largest face
      """ bbox[2] - bbox[0] = width  of face box
          bbox[3] - bbox[1] = height of face box
          width × height    = area   of face box """
      face  = faces[areas.index(max(areas))]

      aligned = face_align.norm_crop(img, landmark = face.kps , image_size = image_size)

      cv2.imwrite(os.path.join(person_dst_path, img_name),aligned)

      saved+=1

  return skipped,total,saved

print("🤞Function created")

In [ ]:
# print("Creating aligned for TRAIN...")

# skipped,total,saved = align_images(train_data,aligned_train,aligned_app)

# print(f"\n  Total images : {total}")
# print(f"  Faces saved  : {saved}")
# print(f"  Skipped      : {len(skipped)}")
# if skipped:
#     for s in skipped:
#         print(f"    {s['person']:30s} | {s['reason']}")

In [ ]:
print("Processing VAL set...")
skipped,total,saved = align_images(val_data, aligned_val, aligned_app)

print(f"\n  Total images : {total}")
print(f"  Faces saved  : {saved}")
print(f"  Skipped      : {len(skipped)}")
if skipped:
    for s in skipped:
        print(f"    {s['person']:30s} | {s['reason']}")

In [ ]:
def visualize_aligned(aligned_dir, samples=3):
    persons = sorted(os.listdir(aligned_dir))
    for person in persons:
        person_path = os.path.join(aligned_dir, person)
        imgs_for_person = [f for f in os.listdir(person_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        # Limit to the number of samples requested
        display_imgs = imgs_for_person[:samples]

        if not display_imgs:
            print(f"No images to display for {person}")
            continue

        fig, axes = plt.subplots(1, len(display_imgs), figsize=(len(display_imgs) * 3, 3))

        # Adjust axes to be iterable even if there's only one image
        if len(display_imgs) == 1:
            axes = [axes]

        fig.suptitle(f"{person[:25]} - Aligned Samples", fontsize=12) # Use suptitle for the person's name

        for i, img_name in enumerate(display_imgs):
            img_path = os.path.join(person_path, img_name)
            img      = cv2.imread(img_path)

            ax = axes[i] # Access the correct subplot axis
            if img is not None:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                ax.set_title(f"Image {i+1}", fontsize=8) # Set individual title if needed
            else:
                ax.set_title("Error", fontsize=8)
            ax.axis('off')

        plt.tight_layout()
        plt.show()

visualize_aligned(aligned_train)

# ***Data Augmentation***

In [ ]:
lfw_persons = ["Jacques_Chirac", "Jean_Chretien",
               "John_Ashcroft", "Junichiro_Koizumi", "Serena_Williams"]

print("Current image counts:\n")
for person in sorted(os.listdir(aligned_train)):
    person_path = os.path.join(aligned_train, person)
    count = len([f for f in os.listdir(person_path)
                 if f.lower().endswith(('.jpg','.jpeg','.png'))])

    if person in lfw_persons:
        print(f"  {person:35s} → {count:3d} images  ⏭️ skip (LFW)")
    else:
        needed = max(0, 100 - count)
        print(f"  {person:35s} → {count:3d} images  ⚠️ need {needed} more")

In [ ]:
import random

def augment_images(img):
  aug = img.copy()
  choise = random.randint(0,3)

  if choise == 0:
    aug = cv2.flip(aug,1) # Horizintal flip

  elif choise == 1:
    alpha = random.uniform(0.6,1.4)
    aug = cv2.convertScaleAbs(aug,alpha=alpha) # Brightness

  elif choise == 2:
    height,width = aug.shape[0:2]
    angle = random.uniform(-13,13)
    rotation_matrix = cv2.getRotationMatrix2D((width//2,height//2),angle,1)
    aug = cv2.warpAffine(aug,rotation_matrix,(width,height))

  elif choise == 3:
    kernal   = random.choice([3, 5])
    aug = cv2.GaussianBlur(aug, (kernal, kernal), 0)  #Blur

  return aug
print("🤞Function Created")

In [ ]:
target = 100

for person in sorted(os.listdir(aligned_train)):
    person_path = os.path.join(aligned_train, person)
    if not os.path.isdir(person_path):
       continue
    if person in lfw_persons:
       continue

    images = [f for f in os.listdir(person_path)
              if f.lower().endswith(('.jpg','.jpeg','.png'))]

    current = len(images)
    needed  = target - current

    if needed <= 0:
        print(f"  ✅ {person:35s} → already {current} images")
        continue

    counter = 0
    while counter < needed:
        # Pick random source image
        src_name = random.choice(images)
        src_path = os.path.join(person_path, src_name)
        img      = cv2.imread(src_path)

        if img is None:
            continue

        # Augment
        aug = augment_images(img)

        # Save with new name
        aug_name = f"aug_{counter:04d}.jpg"
        cv2.imwrite(os.path.join(person_path, aug_name), aug)
        counter += 1

    final_count = len([f for f in os.listdir(person_path)
                       if f.lower().endswith(('.jpg','.jpeg','.png'))])
    print(f"  ✅ {person:35s} → {current} → {final_count} images (+{needed})")

print("\n✅ Augmentation complete!")

In [ ]:
total = 0
for person in sorted(os.listdir(aligned_train)):
    person_path = os.path.join(aligned_train, person)
    if not os.path.isdir(person_path):
      continue
    count = len([f for f in os.listdir(person_path)
                 if f.lower().endswith(('.jpg','.jpeg','.png'))])
    total += count
    status = "✅" if count >= 100 or person in lfw_persons else "⚠️"
    print(f"  {status} {person:35s} → {count:3d} images")

print(f"\n  Total train images after augmentation: {total}")

# ***Face Recognetion***

In [ ]:
arcface_app = FaceAnalysis(name='buffalo_sc',
                        providers=['CPUExecutionProvider'])
arcface_app.prepare(ctx_id=0, det_size=(112, 112))   #ٌِArcFace For alligment

print("✅ ArcFace model loaded successfully")

In [ ]:
# Pick one test image from aligned dataset
test_person   = "Haneen Mohammed Mousa"
test_images   = os.listdir(os.path.join(aligned_train, test_person))
test_img_path = os.path.join(aligned_train, test_person, test_images[0])

# Read image
img = cv2.imread(test_img_path)
# plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
# plt.show()

# Extract embedding directly (skip detector — image already aligned)
embedding = arcface_app.models['recognition'].get_feat(img)

print(f"✅ Embedding extracted successfully")
print(f"   Person    : {test_person}")
print(f"   Image     : {test_images[0]}")
print(f"   Embedding shape : {embedding.shape}")
print(f"   First 5 values  : {embedding[0][:5]}")


In [ ]:
def extract_embeddings(aligned_dir,arcface_app):
  Embeddings = {}    #{person_name: [emb1, emb2, ...]}
  total = 0
  failed = 0

  for person in sorted(os.listdir(aligned_dir)):
    person_path = os.path.join(aligned_dir, person)
    if not os.path.isdir(person_path):
      continue

    images = [f for f in os.listdir(person_path)
              if f.lower().endswith(('.jpg','.jpeg','.png'))]
    embeddings = []
    for img_name in images:
      total += 1
      img_path = os.path.join(person_path, img_name)
      img      = cv2.imread(img_path)

      if img is None:
          failed += 1
          continue

      embedding = arcface_app.models['recognition'].get_feat(img)
      embeddings.append(embedding[0]) # 512

    Embeddings[person] = embeddings
    print(f"  ✅ {person:35s} → {len(embeddings):3d} embeddings")

    print(f"\n  Total processed : {total}")
    print(f"  Total failed    : {failed}")

  return Embeddings,total,failed

print("🤞Function created")


In [ ]:
print("Extracting TRAIN embeddings...")
train_embeddings = extract_embeddings(aligned_train, arcface_app)

In [ ]:
import pickle

# Extract
print("Extracting TRAIN embeddings...")
train_embeddings, total, failed = extract_embeddings(aligned_train, arcface_app)
print(f"\n  Total : {total} | Failed : {failed}")

# Save to Drive
save_path = "/content/drive/MyDrive/embeddings/train_embeddings.pkl"
os.makedirs("/content/drive/MyDrive/embeddings", exist_ok=True)

with open(save_path, 'wb') as f:
    pickle.dump(train_embeddings, f)

print(f"✅ Train embeddings saved to Drive")

In [ ]:
# Extract
print("Extracting VAL embeddings...")
val_embeddings, total, failed = extract_embeddings(aligned_val, arcface_app)

# Save to Drive
save_path = "/content/drive/MyDrive/embeddings/val_embeddings.pkl"

with open(save_path, 'wb') as f:
    pickle.dump(val_embeddings, f)

print(f"✅ Val embeddings saved to Drive")


# ***connect to database***

In [ ]:
with open("/content/drive/MyDrive/embeddings/train_embeddings.pkl","rb") as f:
  train_embeddings = pickle.load(f)

with open("/content/drive/MyDrive/embeddings/val_embeddings.pkl","rb") as f:
  val_embeddings = pickle.load(f)


print(f"Train embeddings : {len(train_embeddings)}")
print(f"Val embeddings   : {len(val_embeddings)}")


In [ ]:
def build_database(embeddings_dict):
  database = {}
  for person , embed_lst in embeddings_dict.items():

    #stack all embeddings into matrix
    emb_matrix = np.stack(embed_lst) # shape ( N , 512)

    # Average all embeddings into one
    avg_emb = np.mean(emb_matrix , axis = 0) # shape (512,)

    # Normalize to unit length
    avg_emb = avg_emb / np.linalg.norm(avg_emb)

    # Add to database
    database[person] = avg_emb
    print(f"  ✅ {person:35s} → embedding shape: {avg_emb.shape}")
  return database

print("🤞Function created")

In [ ]:
database = build_database(train_embeddings)
# Save to Drive
db_path = "/content/drive/MyDrive/embeddings/database.pkl"
with open(db_path, 'wb') as f:
    pickle.dump(database, f)

print(f"\n✅ Database saved → {len(database)} persons")


# ***Similarity***

In [ ]:
def find_match(query_embedding , database , threshold = 0.5):
  """
  query_embedding : (512,) from the camera
  database        :{person_name : (512,)}
  threshold       : minimum similarity to be considered a match

  return : (person_name , similarity) or ("unknown",score)

  """

  best_match = "Unknown"
  best_score = -1

  #Normalize query_embedding
  query = query_embedding / np.linalg.norm(query_embedding)

  for person , emb in database.items():

    # Cosine similarity
    similarity = np.dot(query,emb)
    if similarity > best_score:
      best_score = similarity
      best_match = person


  if best_score  < threshold :
    return "Unknown" , best_score
  else:
    return best_match , best_score

print("🤞Function created")

- load database

In [ ]:
with open ("/content/drive/MyDrive/embeddings/database.pkl" , "rb") as f:
  database = pickle.load(f)

print(f"✅ Database loaded → {len(database)} persons")
for person in sorted(database.keys()):
    print(f"   {person}")

-  Test Matching on Val Set

In [ ]:
print("Testing matching on VAL set...\n")

correct   = 0
wrong     = 0
unknown   = 0
total     = 0

results = []
aligned_val = "/content/drive/MyDrive/aligned_dataset/val"
for person in sorted(os.listdir(aligned_val)):
    person_path = os.path.join(aligned_val, person)
    if not os.path.isdir(person_path): continue

    images = [f for f in os.listdir(person_path)
              if f.lower().endswith(('.jpg','.jpeg','.png'))]

    person_correct = 0
    person_wrong   = 0
    person_unknown = 0

    for img_name in images:
        total += 1
        img_path = os.path.join(person_path, img_name)
        img      = cv2.imread(img_path)
        if img is None: continue

        # Extract embedding
        emb   = arcface_app.models['recognition'].get_feat(img)
        query = emb[0]

        # Match against database
        pred_name, score = find_match(query, database, threshold=0.5)

        if pred_name == person:
            correct += 1
            person_correct += 1
        elif pred_name == "Unknown":
            unknown += 1
            person_unknown += 1
        else:
            wrong += 1
            person_wrong += 1

    status = "✅" if person_wrong == 0 else "⚠️"
    print(f"  {status} {person:35s} → correct: {person_correct:3d} | wrong: {person_wrong:3d} | unknown: {person_unknown:3d}")

print(f"\n{'='*60}")
print(f"  Total images  : {total}")
print(f"  Correct       : {correct}  ({correct/total*100:.1f}%)")
print(f"  Wrong         : {wrong}   ({wrong/total*100:.1f}%)")
print(f"  Unknown       : {unknown}  ({unknown/total*100:.1f}%)")
print(f"  Accuracy      : {correct/total*100:.1f}%")

- Tunning for threshold

In [ ]:
thresholds = [0.3,0.4,0.6,0.7]
for thresh in thresholds:
    correct = wrong = unknown = 0

    for person in sorted(os.listdir(aligned_val)):
        person_path = os.path.join(aligned_val, person)
        if not os.path.isdir(person_path): continue

        images = [f for f in os.listdir(person_path)
                  if f.lower().endswith(('.jpg','.jpeg','.png'))]

        for img_name in images:
            img_path = os.path.join(person_path, img_name)
            img      = cv2.imread(img_path)
            if img is None: continue

            emb              = arcface_app.models['recognition'].get_feat(img)
            pred_name, score = find_match(emb[0], database, threshold=thresh)

            if pred_name == person:   correct += 1
            elif pred_name == "Unknown": unknown += 1
            else:                     wrong   += 1

    total = correct + wrong + unknown
    print(f"  Threshold {thresh} → Accuracy: {correct/total*100:.1f}% | Wrong: {wrong} | Unknown: {unknown}")


In [ ]:
import random

# Pick 3 random persons from val set
test_persons = random.sample(sorted(os.listdir(aligned_val)), 3)

for person in test_persons:
    person_path = os.path.join(aligned_val, person)
    images      = [f for f in os.listdir(person_path)
                   if f.lower().endswith(('.jpg','.jpeg','.png'))]

    # Pick random image
    img_name = random.choice(images)
    img_path = os.path.join(person_path, img_name)
    img      = cv2.imread(img_path)
    img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Extract & match
    emb            = arcface_app.models['recognition'].get_feat(img)
    pred_name, score = find_match(emb[0], database, threshold=0.3)

    # Display result
    color  = (0,255,0) if pred_name == person else (255,0,0)
    status = "✅ CORRECT" if pred_name == person else "❌ WRONG"

    plt.figure(figsize=(3,3))
    plt.imshow(img_rgb)
    plt.title(f"True : {person[:15]}\nPred : {pred_name[:15]}\nScore: {score:.3f}  {status}",
              fontsize=9)
    plt.axis('off')
    plt.show()